In [ ]:
import wandb
api = wandb.Api()
run = api.run("/vlm_rl/rl_train/runs/l7wr4tyn")


wandb: Currently logged in as: axi42 (vlm_rl) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


          spl  train/entropy      episode_label  n_steps  _step  throughput  \
0    0.579565       0.558594  qgZhhx1MpTi_14138       23      0    5.389591   
1    0.000000       0.679688   8wJuSPJ9FXG_4074       34      1    5.817367   
2    0.000000       0.474609  DqJKU7YU7dA_22410      283      7    4.613979   
3    0.262774       0.429688  RTV2n6fXB2w_16859      186      8    5.278451   
4    0.000000       0.554688   XiJhRLvpKpX_3225      268      9    5.630786   
..        ...            ...                ...      ...    ...         ...   
495  0.227125       0.462891    WhNyDTnd9g5_437       46   1365    7.919921   
496  0.206202       0.365234  kA2nG18hCAr_21671       86   1366    4.867042   
497  0.849102       0.357422  NEVASPhcrxR_16522       55   1370    6.824809   
498  0.000000       0.468750  92vYG1q49FY_52590      259   1371    4.501245   
499  0.000000       0.472656  zepmXAdrpjR_13775       65   1377    6.895378   

     worker_pid  loss/pg_loss  sup/min_vlm_latency 

In [60]:
import pickle
result_file = "dump/rl_training/ovon_rpp_ke_g97/dbg/result_0.pkl"
with open(result_file,"rb") as f:
    results = pickle.load(f)

In [62]:
results

[{'distance_to_goal': 2.983255386352539,
  'success': 0.0,
  'spl': 0.0,
  'soft_spl': 0.02450227195351482,
  'pos_rots': [-7.63953161239624,
   2.719700574874878,
   3.3554728031158447,
   0.0,
   -0.8772289752960205,
   0.0,
   -0.4800722897052765],
  'scene_id': 'XfUxBGTFQQb',
  'episode_id': '5421',
  'episode_label': 'XfUxBGTFQQb_5421',
  'epoch': 0,
  'step_in_epoch': 90,
  'oracle_action': -1,
  'steps': 187,
  'instr_or_goal': 'cabinet'},
 {'distance_to_goal': 0.22379766404628754,
  'success': 1.0,
  'spl': 0.37108168146725684,
  'soft_spl': 0.35434502063727896,
  'pos_rots': [5.742013931274414,
   0.05998396873474121,
   1.9132561683654785,
   0.0,
   -0.38196587562561035,
   0.0,
   0.9241764545440674],
  'scene_id': 'yX5efd48dLf',
  'episode_id': '46555',
  'episode_label': 'yX5efd48dLf_46555',
  'epoch': 0,
  'step_in_epoch': 86,
  'oracle_action': 0,
  'steps': 106,
  'instr_or_goal': 'cloth'},
 {'distance_to_goal': 4.236571788787842,
  'success': 0.0,
  'spl': 0.0,
  'sof

In [ ]:
df = run.history()


'goal'

In [ ]:
df.groupby('goal').size().sort_values(ascending=False)

In [ ]:
df[['goal','rollout/ep_rtn']].groupby('goal').mean().sort_values(ascending=False,by='rollout/ep_rtn')

In [43]:
final_df = df.groupby('goal').agg(
    mean_rtn=('rollout/ep_rtn', 'mean'),
    sample_count=('rollout/ep_rtn', 'count')
).sort_values(by='mean_rtn', ascending=False).query('sample_count>4')

In [44]:
# specific_goals is the index from your previous filtered result
valid_goals = final_df.index

# Filter the raw data to only include these groups
filtered_raw_df = df[df['goal'].isin(valid_goals)].copy()

In [46]:
# Broadcast group means back to the original rows
filtered_raw_df['group_mean'] = filtered_raw_df.groupby('goal')['rollout/ep_rtn'].transform('mean')

# Calculate Squared Error (SE) for each row
filtered_raw_df['squared_error'] = (filtered_raw_df['rollout/ep_rtn'] - filtered_raw_df['group_mean']) ** 2

In [50]:
filtered_raw_df['squared_error'].mean()

1.2509536239363492

In [51]:
np.var(filtered_raw_df['rollout/ep_rtn'])

1.516929989355018

In [54]:
df['goal'].unique()

array(['towel', 'table', 'balustrade', 'washbasin', 'pool table',
       'kitchen lower cabinet', 'display cabinet', 'tv', 'wardrobe',
       'window', 'couch', 'storage', 'bathroom towel', 'stove', 'jacket',
       'bathroom cabinet', 'blanket', 'cloth', 'bar', 'pillar', 'chair',
       'window shutter', 'bathtub', 'rack', 'desk', 'board', 'countertop',
       'bench', 'sleeping bag', 'bathroom accessory', 'trashcan',
       'foot spa', 'dinner table', 'washing machine', 'oven', 'cabinet',
       'bar cabinet', 'duct', 'oven and stove', 'stool', 'range hood',
       'sheet', 'sink', 'clothes', 'cabinet clutter', 'toilet', 'hose',
       'monitor', 'whiteboard', 'kitchen appliance', 'ladder', 'plate',
       'trampoline', 'sofa seat', 'patio chair', 'bed', 'swivel chair',
       'basket', 'sink cabinet', 'headboard', 'pouffe', 'treadmill',
       'box', 'bottle', 'drawer', 'screen', 'bed curtain',
       'stack of papers', 'bathroom shelf', 'air conditioner',
       'storage shelving',

In [57]:
# Requires transformers>=4.51.0

import torch
import torch.nn.functional as F

from torch import Tensor
from transformers import AutoTokenizer, AutoModel


def last_token_pool(last_hidden_states: Tensor,
                 attention_mask: Tensor) -> Tensor:
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    else:
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = last_hidden_states.shape[0]
        return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths]


def get_detailed_instruct(task_description: str, query: str) -> str:
    return f'Instruct: {task_description}\nQuery:{query}'

# Each query must come with a one-sentence instruction that describes the task
task = 'Given an object, retrieve the most similar object'

queries = [
    get_detailed_instruct(task, 'cat'),
    get_detailed_instruct(task, 'grape')
]
# No need to add instruction for retrieval documents
documents = [
    "apple",
    "dog",
    "orange"
]
input_texts = queries + documents

tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-Embedding-0.6B', padding_side='left')
model = AutoModel.from_pretrained('Qwen/Qwen3-Embedding-0.6B')

# We recommend enabling flash_attention_2 for better acceleration and memory saving.
# model = AutoModel.from_pretrained('Qwen/Qwen3-Embedding-0.6B', attn_implementation="flash_attention_2", torch_dtype=torch.float16).cuda()

max_length = 8192

# Tokenize the input texts
batch_dict = tokenizer(
    input_texts,
    padding=True,
    truncation=True,
    max_length=max_length,
    return_tensors="pt",
)
batch_dict.to(model.device)
outputs = model(**batch_dict)
embeddings = last_token_pool(outputs.last_hidden_state, batch_dict['attention_mask'])

# normalize embeddings
embeddings = F.normalize(embeddings, p=2, dim=1)
scores = (embeddings[:2] @ embeddings[2:].T)
print(scores.tolist())
# [[0.7645568251609802, 0.14142508804798126], [0.13549736142158508, 0.5999549627304077]]


[[0.42088836431503296, 0.5814238786697388, 0.4988881051540375], [0.41622915863990784, 0.4083576798439026, 0.49350351095199585]]


In [55]:
%pip install sentense_transformers

ERROR: Could not find a version that satisfies the requirement sentense_transformers (from versions: none)
ERROR: No matching distribution found for sentense_transformers
Note: you may need to restart the kernel to use updated packages.


In [23]:
import pandas as pd

# Set options to display all rows and columns
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None) # Also set max column width if needed
pd.set_option('display.width', 1000)